In [ ]:
%%bash
pip install -q -U google-genai httpx ddgs pypdf openpyxl pandas playwright

In [ ]:
import os
import sys
import json
import inspect
from google.colab import drive
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY", "")

if not api_key:
    api_key = input("Nhap GEMINI_API_KEY: ").strip()

assert api_key, "Thieu GEMINI_API_KEY"
client = genai.Client(api_key=api_key, http_options={'api_version': 'v1beta'})
MODEL_ID = "gemini-3.5-flash"
print("Khoi tao Gemini Client thanh cong voi model:", MODEL_ID)

drive.mount("/content/drive", force_remount=False)

BASE_DIR = "/content/drive/MyDrive/AutoEvolve_AI"
SKILLS_DIR = f"{BASE_DIR}/skills"

for p in (BASE_DIR, SKILLS_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

assert os.path.isdir(SKILLS_DIR), f"Khong tim thay thu muc skills tai: {SKILLS_DIR}"
print("Da ket noi kho skills:", sorted(f for f in os.listdir(SKILLS_DIR) if f.endswith(".py") and not f.startswith("test_")))

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
except Exception:
    api_key = os.environ.get("GEMINI_API_KEY", "")

if not api_key:
    api_key = input("Nhap GEMINI_API_KEY: ").strip()

assert api_key, "Thieu GEMINI_API_KEY"
client = genai.Client(api_key=api_key, http_options={'api_version': 'v1beta'})
MODEL_ID = "gemini-3.5-flash"
print("Khoi tao Gemini Client thanh cong voi model:", MODEL_ID)

In [4]:
SKIP_MODULES = {
    "skill_circular_buffer",
    "skill_event_processor",
    "skill_event_queue",
    "skill_thread_pool",
    "skill_url_deduplicator_processor",
    "skill_url_filter",
    "skill_url_normalizer",
    "skill_url_normalizer_filter_pipeline",
    "skill_url_processor_pipeline",
    "skill_html_content_extractor",
    "skill_html_noise_filter",
    "skill_json_schema_validator",
}


def _safe_serialize(obj, max_chars: int = 8000) -> str:
    try:
        raw = json.dumps(obj, ensure_ascii=False, default=str)
    except Exception:
        raw = str(obj)
    if len(raw) > max_chars:
        return raw[:max_chars] + f"... [cat bot, tong {len(raw)} ky tu]"
    return raw


def _ann_to_schema_type(ann) -> types.Type:
    """Chuyển type annotation bất kỳ (kể cả inspect._empty hoặc Union) sang types.Type hợp lệ."""
    if ann is inspect._empty or ann is None:
        return types.Type.STRING

    origin = getattr(ann, "__origin__", None)
    args = getattr(ann, "__args__", ())

    if origin is not None:
        # Xử lý Optional / Union: lấy kiểu đầu tiên không phải NoneType
        non_none = [a for a in args if a is not type(None)]
        if len(args) == 2 and len(non_none) == 1:
            return _ann_to_schema_type(non_none[0])
        if origin in (list, tuple, set):
            return types.Type.ARRAY
        if origin is dict:
            return types.Type.OBJECT

    if ann is int:
        return types.Type.INTEGER
    if ann is float:
        return types.Type.NUMBER
    if ann is bool:
        return types.Type.BOOLEAN
    if ann in (list, tuple, set):
        return types.Type.ARRAY
    if ann is dict:
        return types.Type.OBJECT
    return types.Type.STRING


def _default_for_type(schema_type: types.Type, param: inspect.Parameter):
    if param.default is not inspect._empty:
        return param.default
    if schema_type == types.Type.INTEGER:
        return 5
    if schema_type == types.Type.NUMBER:
        return 1.0
    if schema_type == types.Type.BOOLEAN:
        return False
    if schema_type == types.Type.ARRAY:
        return []
    if schema_type == types.Type.OBJECT:
        return {}
    return ""


def build_gemini_tools(skills_dir: str = SKILLS_DIR):
    """Tạo danh sách (FunctionDeclaration, callable_map) để chạy AFC mượt trên mọi skill."""
    declarations = []
    handlers = {}
    loaded_names = []

    for fname in sorted(os.listdir(skills_dir)):
        if not fname.startswith("skill_") or not fname.endswith(".py") or fname.startswith("test_"):
            continue
        mod_name = fname[:-3]
        if mod_name in SKIP_MODULES:
            continue

        try:
            mod = __import__(mod_name)
        except Exception as e:
            print(f"[SKIP] Loi import {mod_name}: {e}")
            continue

        for attr_name, obj in inspect.getmembers(mod, inspect.isclass):
            if obj.__module__ != mod_name:
                continue
            for m_name, method in inspect.getmembers(obj, predicate=inspect.isfunction):
                if m_name.startswith("_"):
                    continue

                sig = inspect.signature(method)
                tool_fn_name = f"{mod_name.replace('skill_', '')}__{m_name}"

                properties = {}
                required = []
                param_meta = []

                for p_name, p in sig.parameters.items():
                    if p_name == "self" or p.kind in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD):
                        continue
                    s_type = _ann_to_schema_type(p.annotation)
                    has_default = p.default is not inspect._empty
                    desc_p = f"Tham so `{p_name}` (mac dinh: {p.default})" if has_default else f"Tham so `{p_name}`"
                    properties[p_name] = types.Schema(type=s_type, description=desc_p)
                    if not has_default:
                        required.append(p_name)
                    param_meta.append((p_name, s_type, p))

                doc_first = (
                    inspect.getdoc(method)
                    or inspect.getdoc(obj)
                    or f"Goi {cls_ref_name}.{m_name}" if (cls_ref_name := obj.__name__) else ""
                ).splitlines()[0]

                decl = types.FunctionDeclaration(
                    name=tool_fn_name,
                    description=f"[{mod_name}] {doc_first}",
                    parameters=types.Schema(
                        type=types.Type.OBJECT,
                        properties=properties,
                        required=required,
                    ),
                )

                def _make_handler(cls_ref, fn_name, p_list):
                    def _run(**kwargs):
                        clean_kwargs = {}
                        for name, s_type, p in p_list:
                            if name in kwargs and kwargs[name] is not None:
                                val = kwargs[name]
                                try:
                                    if s_type == types.Type.INTEGER:
                                        val = int(val)
                                    elif s_type == types.Type.NUMBER:
                                        val = float(val)
                                    elif s_type == types.Type.BOOLEAN and isinstance(val, str):
                                        val = val.lower() in ("true", "1", "yes")
                                except Exception:
                                    pass
                                clean_kwargs[name] = val
                            elif p.default is not inspect._empty:
                                clean_kwargs[name] = p.default

                        try:
                            instance = cls_ref()
                            result = getattr(instance, fn_name)(**clean_kwargs)
                            return {"ok": True, "data": json.loads(_safe_serialize(result))}
                        except Exception as exc:
                            return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}

                    return _run

                declarations.append(decl)
                handlers[tool_fn_name] = _make_handler(obj, m_name, param_meta)
                loaded_names.append(tool_fn_name)

    return declarations, handlers, loaded_names


gemini_declarations, gemini_handlers, loaded_tool_names = build_gemini_tools()
print(f"Da nap {len(gemini_declarations)} tool cho Gemini:")
for n in loaded_tool_names:
    print("  -", n)

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Da nap 49 tool cho Gemini:
  - arxiv_api_client__search_papers
  - binance_crypto_client__get_crypto_price
  - browser_reader__extract_elements
  - browser_reader__read_dynamic_page
  - document_analyzer__analyze_tabular
  - document_analyzer__read_pdf
  - frankfurter_forex_client__get_exchange_rate
  - github_api_client__search_repositories
  - hacker_news_api_client__get_story
  - hacker_news_api_client__get_top_stories
  - huggingface_api_client__search_models
  - network_inspector__inspect_url
  - open_meteo_weather_client__get_current_weather
  - pypi_api_client__get_package_info
  - rss_feed_reader__get_latest_articles
  - weather_api_client__get_current_weather
  - weather_api_client__get_latitude
  - weather_api_client__get_longitude
  - web_reader__read_url
  - web_reader__check_for_whole_start_tag
  - web_reader__clear_cdata_mode
  - web_reader__close
  - web_reader__feed
  - web_reader__get_starttag_text
  - web_reader__getpos
  - web_reader__goahead
  - web_reader__handle_c

In [5]:
SYSTEM_INSTRUCTION = (
    "Ban la tro ly AI thuc chien tren Colab. "
    "Khi can du lieu thuc te (tin tuc, web, API, gia ca, thoi tiet, tai lieu PDF/CSV/Excel), "
    "hay uu tien goi tool phu hop thay vi doan. "
    "Chi dua tren du lieu tra ve tu tool, khong tu bia link URL hay con so."
)


def ask_gemini_with_tools(prompt: str, model: str = MODEL_ID, temperature: float = 0.2, max_steps: int = 6) -> str:
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        tools=[types.Tool(function_declarations=gemini_declarations)],
        temperature=temperature,
    )
    chat = client.chats.create(model=model, config=config)
    message_input = prompt
    called = []

    for _ in range(max_steps):
        response = chat.send_message(message_input)

        func_calls = []
        for cand in getattr(response, "candidates", []) or []:
            content = getattr(cand, "content", None)
            for part in getattr(content, "parts", []) or []:
                fc = getattr(part, "function_call", None)
                if fc and getattr(fc, "name", None):
                    func_calls.append(fc)

        if not func_calls:
            if called:
                print("[TOOLS CALLED]:", " -> ".join(called))
            return response.text or ""

        response_parts = []
        for fc in func_calls:
            fname = fc.name
            fargs = dict(fc.args or {})
            called.append(fname)
            handler = gemini_handlers.get(fname)
            if handler is None:
                payload = {"ok": False, "error": f"Khong tim thay handler cho tool {fname}"}
            else:
                payload = handler(**fargs)

            response_parts.append(
                types.Part.from_function_response(
                    name=fname,
                    response={"result": _safe_serialize(payload)},
                )
            )

        message_input = response_parts

    if called:
        print("[TOOLS CALLED]:", " -> ".join(called))
    return response.text or ""

In [7]:
# Test 1: Goi skill_document_analyzer doc PDF truc tuyen
print(ask_gemini_with_tools(
    "Đọc 2 trang đầu của bài báo https://arxiv.org/pdf/2508.07407.pdf và tóm tắt ngắn gọn mục tiêu chính."
))

print("-" * 60)

# Test 2: Goi skill_web_search / API client
print(ask_gemini_with_tools(
    "Tìm thông tin mới nhất về phiên bản Python 3.14 hoặc 3.15 trên python.org kèm link nguồn."
))

[TOOLS CALLED]: document_analyzer__read_pdf
Dựa trên nội dung 2 trang đầu của bài báo **"A Comprehensive Survey of Self-Evolving AI Agents: A New Paradigm Bridging Foundation Models and Lifelong Agentic Systems"** (arXiv:2508.07407), dưới đây là tóm tắt ngắn gọn mục tiêu chính:

### **Mục tiêu chính của bài báo**
Bài báo là một **nghiên cứu tổng quan (survey) toàn diện** nhằm hệ thống hóa và định hình lĩnh vực **AI Agent Tự Tiến Hóa (Self-Evolving AI Agents)** — một hướng đi kết nối các mô hình nền tảng tĩnh (static foundation models) với khả năng thích ứng liên tục của hệ thống agent học suốt đời (lifelong agentic systems).

Cụ thể, bài báo hướng tới các mục tiêu:

1. **Giải quyết hạn chế của AI Agent hiện tại:** Khắc phục nhược điểm của các hệ thống agent đơn/đa agent hiện nay — vốn phụ thuộc nhiều vào cấu hình thủ công và giữ nguyên cấu trúc cố định sau khi triển khai, khó thích ứng với môi trường động.
2. **Đề xuất khung lý thuyết thống nhất (Unified Conceptual Framework):** Xây dự

[TOOLS CALLED]: web_search__search -> web_search__search -> web_reader__read_url -> web_search__search -> web_search__search -> web_reader__read_url



In [8]:
# Test 1: Goi skill_document_analyzer doc PDF truc tuyen
print(ask_gemini_with_tools(
    "Tôi đang ỏ đâu và mấy giờ rồi, thời tiết sao"
))



[TOOLS CALLED]: weather_api_client__get_current_weather
Xin lỗi bạn, vì lý do bảo mật và quyền riêng tư, mình **không thể tự động truy cập vị trí địa lý (GPS/IP)** thiết bị của bạn, nên mình chưa thể biết chính xác bạn đang ở đâu.

Để mình có thể xem thời gian và thời tiết chính xác cho bạn, **bạn hãy cho mình biết tên Thành phố / Tỉnh nơi bạn đang ở** (ví dụ: Hà Nội, TP. Hồ Chí Minh, Đà Nẵng, Đà Lạt,...).

---

📌 *Ví dụ dữ liệu thời tiết & thời gian tại **Hà Nội** (cập nhật qua hệ thống):*
* **Thời gian (giờ địa phương):** 08:00
* **Nhiệt độ:** 19.2°C
* **Tốc độ gió:** 14.3 km/h
* **Trạng thái:** Có mây / Nhiều mây

👉 *Hãy nhắn cho mình biết thành phố của bạn nhé!*
